# 準備演習 02: チーム開発ワークフロー向けの Claude Code 設定

## 目的

- project-level / subdir-level の `CLAUDE.md` 階層を手を動かして理解する
- `.claude/rules/` のパス固有ルール、`.claude/skills/` の `context: fork`、MCP 設定を順番に観察する
- カスタムスラッシュコマンド、plan mode、direct execution の使い分けを学ぶ

## 対象ドメイン

- Domain 3: Claude Code Configuration & Workflows

## 完成イメージ

この Notebook は **設定ファイルを書いて観察する教材** です。runtime 実装よりも、ルールの適用スコープと運用判断を理解することを優先します。
必要に応じて、完成版 Lab [../labs/02-claude-code-team-workflow/](../labs/02-claude-code-team-workflow/) と Claude Code 公式ドキュメントを参照してください。

In [ ]:
from pathlib import Path
import json
import tempfile
import textwrap

workspace = Path(tempfile.mkdtemp(prefix="claude-code-prep-"))
section = lambda title: print(f"\n=== {title} ===")

section("sandbox workspace")
print(workspace)

## Step 1. project-level の `CLAUDE.md` を作る

チーム全員に一貫して適用したいルールは project-level に置きます。

In [ ]:
project_claude = textwrap.dedent("""
# Team standards

- Python は型ヒントを付ける
- 機能変更時は関連テストを更新する
- 危険な変更は plan mode で計画を提示してから進める
""").strip()

src_claude = textwrap.dedent("""
# src-specific guidance

- src/ 配下ではドメインロジックを薄い関数に分割する
- API 境界では入出力スキーマを明示する
""").strip()

(workspace / "src").mkdir(parents=True, exist_ok=True)
(workspace / "CLAUDE.md").write_text(project_claude, encoding="utf-8")
(workspace / "src" / "CLAUDE.md").write_text(src_claude, encoding="utf-8")

hierarchy = [
    "~/.claude/CLAUDE.md (global)",
    str(workspace / "CLAUDE.md"),
    str(workspace / "src" / "CLAUDE.md"),
]

section("CLAUDE.md hierarchy")
print("優先順位:")
for item in hierarchy:
    print("-", item)
print("\nproject-level:")
print((workspace / "CLAUDE.md").read_text(encoding="utf-8"))
print("\nsubdir-level:")
print((workspace / "src" / "CLAUDE.md").read_text(encoding="utf-8"))

### 確認ポイント

- コーディング標準とテスト規約のような **全員共通ルール** を project-level に置けているか
- リポジトリ全体に適用される前提を説明できるか

## Step 2. `.claude/rules/` でパス固有ルールを追加する

glob にマッチしたファイル編集時だけ追加ルールが読み込まれる想定を、簡易シミュレーションで確認します。

In [ ]:
rules_dir = workspace / ".claude" / "rules"
rules_dir.mkdir(parents=True, exist_ok=True)

python_rule = textwrap.dedent("""
---
globs: ["src/**/*.py"]
---
Python 実装では logging を print より優先する。
""").strip()

test_rule = textwrap.dedent("""
---
globs: ["tests/**/*.py"]
---
テストでは arrange-act-assert を明示する。
""").strip()

api_rule = textwrap.dedent("""
---
globs: ["src/api/**/*.py"]
---
API では例外を HTTP エラーへ正規化する。
""").strip()

(rules_dir / "python.md").write_text(python_rule, encoding="utf-8")
(rules_dir / "tests.md").write_text(test_rule, encoding="utf-8")
(rules_dir / "api.md").write_text(api_rule, encoding="utf-8")

from pathlib import PurePosixPath

def matched_rules(path: str):
    path_obj = PurePosixPath(path)
    matches = []
    for file in sorted(rules_dir.glob("*.md")):
        header = file.read_text(encoding="utf-8").split("---")
        if len(header) < 3:
            continue
        globs = [part.strip().strip('"') for part in header[1].split("[")[-1].split("]")[0].split(",") if part.strip()]
        if any(path_obj.match(pattern) for pattern in globs):
            matches.append(file.name)
    return matches

section("rule matching simulation")
for target in ["src/app/service.py", "src/api/orders/refund.py", "tests/unit/test_refund.py"]:
    print(target, "->", matched_rules(target))

## Step 3. カスタムスラッシュコマンドとスキルを分ける

よく使う定型タスクは `.claude/commands/`、より大きな作業テンプレートは `.claude/skills/` に置くと整理しやすくなります。

In [ ]:
commands_dir = workspace / ".claude" / "commands"
skills_dir = workspace / ".claude" / "skills" / "refactor-service"
commands_dir.mkdir(parents=True, exist_ok=True)
skills_dir.mkdir(parents=True, exist_ok=True)

slash_command = textwrap.dedent("""
---
description: 変更差分のレビュー観点を列挙する
---
1. 変更ファイルを整理して要約してください
2. テスト不足とリスクを洗い出してください
""").strip()

skill = textwrap.dedent("""
---
name: refactor-service
context: fork
allowed-tools: [Read, Grep, Bash]
---
既存サービスの責務を分割し、リスクを箇条書きで返す。
""").strip()

(commands_dir / "review-change.md").write_text(slash_command, encoding="utf-8")
(skills_dir / "SKILL.md").write_text(skill, encoding="utf-8")

section("command vs skill")
print((commands_dir / "review-change.md").read_text(encoding="utf-8"))
print("---")
print((skills_dir / "SKILL.md").read_text(encoding="utf-8"))

### 確認ポイント

- slash command は会話内の定型操作
- skill は `context: fork` と `allowed-tools` で **分離実行** を明示できる
- main context を汚さずに重い調査や移行計画を切り出せる

## Step 4. project-level MCP 設定を追加し、個人設定と併存させる

In [ ]:
project_mcp = {
    "mcpServers": {
        "github": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-github"],
            "env": {"GITHUB_TOKEN": "${GITHUB_TOKEN}"},
        }
    }
}

personal_mcp = {
    "mcpServers": {
        "local-db": {
            "command": "python",
            "args": ["/Users/me/bin/local_db_server.py"],
        }
    }
}

(workspace / ".mcp.json").write_text(json.dumps(project_mcp, ensure_ascii=False, indent=2), encoding="utf-8")

merged_server_names = sorted(set(project_mcp["mcpServers"]) | set(personal_mcp["mcpServers"]))
section("project MCP + personal MCP coexistence")
print("project file:")
print((workspace / ".mcp.json").read_text(encoding="utf-8"))
print("merged server names:", merged_server_names)

## Step 5. plan mode と direct execution を比較する

複雑さが違う 3 ケースを並べ、どのモードが適切かを判断します。

In [ ]:
CASES = [
    {"task": "単一ファイルの typo 修正", "risk": "low", "recommended": "direct execution"},
    {"task": "src/ 配下の命名規則をまとめて移行", "risk": "medium", "recommended": "plan mode"},
    {"task": "複数案のある新機能追加", "risk": "high", "recommended": "plan mode"},
]

section("mode selection")
for case in CASES:
    print(f"- {case['task']}: {case['recommended']} ({case['risk']} risk)")

## 完成版 Lab 参照

- Notebook では一時 workspace に設定を書き出し、**何がどこに効くのか** を観察しました
- 完成版 Lab では `CLAUDE.md`、`.claude/rules/`、`.claude/skills/`、`.mcp.json` のサンプルが整理されています
- 詳細は [../labs/02-claude-code-team-workflow/](../labs/02-claude-code-team-workflow/) を参照してください
- Claude Code / Agent SDK の最新仕様は公式ドキュメントで確認し、必要に応じて設定例を更新してください